# Lab 5 — PyTorch from scratch: tensors, autograd and a first neural network

**Deep Learning for Computer Vision — Day 2** · ECAM, 5MAU · 2026–2027

You already know the principles of machine learning (loss, gradient descent, train/validation split, overfitting). Today we use **PyTorch**, the library used in research and increasingly in industry, to build and train neural networks.

**Goals.**
1. Manipulate tensors, and convert between OpenCV images `(H, W, C)` and PyTorch tensors `(C, H, W)`.
2. Understand **autograd**: PyTorch computes the gradients for you — check one by hand.
3. Write the canonical **training loop** and an **evaluation** function.
4. Compare a linear classifier (= multinomial logistic regression) with a multi-layer perceptron (MLP) on Fashion-MNIST.
5. Observe overfitting on a small training set.

**Duration:** ~60 min. Training runs on a laptop CPU in a few seconds per epoch.

> *References:* PyTorch tutorials, https://pytorch.org/tutorials/ · KenN7, *Deep learning for computer vision*, ECAM, labs 1–4 (https://github.com/KenN7/deep-learning, MIT licence) · A. Zhang *et al.*, *Dive into Deep Learning*, ch. 2–5, https://d2l.ai/ · H. Xiao, K. Rasul, R. Vollgraf, *Fashion-MNIST*, arXiv:1708.07747, 2017.

In [ ]:
# Setup check: are all the packages of requirements.txt installed in this kernel?
import importlib
import importlib.util

importlib.invalidate_caches()
REQUIRED = {"torch": "torch", "torchvision": "torchvision", "numpy": "numpy", "matplotlib": "matplotlib",
            "cv2": "opencv-contrib-python", "ipywidgets": "ipywidgets"}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
assert not missing, (f"Missing packages: {missing}. Run `%pip install -r ../requirements.txt` "
                     "in a cell, then restart the kernel (Kernel > Restart) and run the notebook again.")
print("Setup OK: all required packages are installed.")

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

DATA = Path("data") if Path("data").exists() else Path("../data")
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "| device:", device)

## 1. Tensors

A `torch.Tensor` is a NumPy-like n-dimensional array that can live on a GPU and remember how it was computed (for autograd).

In [ ]:
a = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print(a.shape, a.dtype, a.device)
print(a @ a.T)              # matrix product
print(a.mean(dim=0))        # reduction along an axis
print(a.numpy())            # back to NumPy (shares memory on CPU)
b = torch.from_numpy(np.arange(6).reshape(2, 3))
print(b.dtype)              # int64: dtype is kept from NumPy!

### Exercise 1.1 — Images: OpenCV ↔ PyTorch
OpenCV images are `uint8` arrays of shape `(H, W, C)` in **BGR** order. PyTorch vision models expect `float32` tensors of shape `(C, H, W)` in **RGB**, values in `[0, 1]`, usually with a batch dimension in front: `(N, C, H, W)`.

Write `to_tensor(img_bgr)` and `to_image(tensor)` (inverse). Hint: `permute`, `unsqueeze`, `float() / 255`.

In [ ]:
img = (np.random.default_rng(0).random((48, 64, 3)) * 255).astype(np.uint8)  # a fake BGR image


def to_tensor(img_bgr):
    # TODO
    ...


def to_image(t):
    # TODO
    ...


x = to_tensor(img)
print(x.shape, x.dtype, float(x.min()), float(x.max()))
assert x.shape == (1, 3, 48, 64) and x.dtype == torch.float32
assert np.isclose(x[0, 0, 5, 7].item(), img[5, 7, 2] / 255)  # channel 0 = red = BGR index 2
assert np.array_equal(to_image(x), img)

## 2. Autograd

Every operation on a tensor with `requires_grad=True` is recorded in a **computational graph**. `loss.backward()` applies the chain rule backwards through this graph and stores $\partial \text{loss} / \partial w$ in `w.grad`.

### Exercise 2.1 — Check a gradient by hand
A single neuron: $z = w x + b$, $y = \mathrm{ReLU}(z)$, loss $L = (y - t)^2$, with $x = 2$, $w = 1.5$, $b = -1$, $t = 1$.
Compute **on paper** $\partial L/\partial w$ and $\partial L/\partial b$ (chain rule), then compare with autograd.

In [ ]:
x, t = torch.tensor(2.0), torch.tensor(1.0)
w = torch.tensor(1.5, requires_grad=True)
b = torch.tensor(-1.0, requires_grad=True)

L = (F.relu(w * x + b) - t) ** 2
L.backward()
print("autograd: dL/dw =", w.grad.item(), " dL/db =", b.grad.item())

# TODO
dL_dw = ... ; dL_db = ...
assert np.isclose(dL_dw, w.grad.item()) and np.isclose(dL_db, b.grad.item())

### Exercise 2.2 — Gradient descent "by hand"
Fit $y = a x + c$ to noisy data by gradient descent, using autograd for the gradients but **writing the update yourself**. Remember:
* gradients **accumulate**: reset them after each step (`p.grad.zero_()`),
* the update itself must not be recorded in the graph (`with torch.no_grad():`).

In [ ]:
xs = torch.linspace(-1, 1, 100)
ys = 3 * xs + 0.5 + 0.2 * torch.randn(100)
a = torch.zeros(1, requires_grad=True)
c = torch.zeros(1, requires_grad=True)
lr = 0.1
for step in range(200):
    loss = ((a * xs + c - ys) ** 2).mean()
    # TODO
    ...
print(f"a = {a.item():.3f} (true 3), c = {c.item():.3f} (true 0.5), loss = {loss.item():.4f}")
assert abs(a.item() - 3) < 0.15 and abs(c.item() - 0.5) < 0.1

`torch.optim` does exactly this for you (and more: momentum, Adam...). The three lines `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()` are at the heart of every training loop.

## 3. The data: Fashion-MNIST

70 000 grey images 28×28 of clothes, 10 classes — same format as MNIST digits but harder. `torchvision` downloads it (≈ 30 MB) the first time.

A `Dataset` returns `(image, label)` pairs; a `DataLoader` groups them into shuffled **mini-batches**.

In [ ]:
tfm = transforms.ToTensor()  # uint8 (H, W) -> float (1, H, W) in [0, 1]
train_full = datasets.FashionMNIST(DATA, train=True, download=True, transform=tfm)
test_set = datasets.FashionMNIST(DATA, train=False, download=True, transform=tfm)
train_set, val_set = torch.utils.data.random_split(train_full, [50000, 10000],
                                                   generator=torch.Generator().manual_seed(0))
classes = train_full.classes
train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=512)
print(len(train_set), "train /", len(val_set), "val /", len(test_set), "test")

xb, yb = next(iter(train_loader))
print("batch:", xb.shape, yb.shape)
fig, ax = plt.subplots(2, 8, figsize=(14, 4))
for k, a_ in enumerate(ax.ravel()):
    a_.imshow(xb[k, 0], cmap="gray"); a_.set_title(classes[yb[k]], fontsize=9); a_.axis("off")

## 4. Two models

* **Linear**: flatten the 784 pixels, one linear layer to 10 scores. With the softmax + cross-entropy loss this is exactly the multinomial **logistic regression** of your ML course.
* **MLP**: two hidden layers with ReLU activations. The non-linearities let the network build intermediate *features*.

`nn.CrossEntropyLoss` takes the raw scores (**logits**) and applies the softmax internally: **do not add a softmax at the end of the model**.

### Exercise 4.1
Complete the `MLP` class: 784 → 256 → 128 → 10 with ReLU.

In [ ]:
class Linear(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(28 * 28, 10)

    def forward(self, x):
        return self.fc(x.flatten(1))  # (N, 1, 28, 28) -> (N, 784) -> (N, 10)


class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO
        ...

    def forward(self, x):
        # TODO
        ...


def n_params(model):
    return sum(p.numel() for p in model.parameters())


print("Linear:", n_params(Linear()), "parameters | MLP:", n_params(MLP()), "parameters")
assert MLP()(torch.zeros(5, 1, 28, 28)).shape == (5, 10)
assert n_params(MLP()) == 784 * 256 + 256 + 256 * 128 + 128 + 128 * 10 + 10

## 5. Training and evaluation loops

### Exercise 5.1
Complete `train_one_epoch` (the 5 canonical steps: forward, loss, zero_grad, backward, step) and `evaluate` (no gradients, `model.eval()`, accuracy).

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        # TODO
        ...
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        # TODO
        ...
    return total_loss / n, correct / n


def fit(model, train_loader, val_loader, epochs, lr=1e-3):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for epoch in range(epochs):
        t0 = time.perf_counter()
        tl, ta = train_one_epoch(model, train_loader, optimizer, loss_fn)
        vl, va = evaluate(model, val_loader, loss_fn)
        for k, v in zip(history, [tl, vl, ta, va]):
            history[k].append(v)
        print(f"epoch {epoch + 1:2d}  train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va:.3f}"
              f"  ({time.perf_counter() - t0:.1f}s)")
    return history


def plot_history(*histories, labels=None):
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    for k, h in enumerate(histories):
        lab = labels[k] if labels else ""
        ax[0].plot(h["train_loss"], "--", color=f"C{k}", label=f"{lab} train")
        ax[0].plot(h["val_loss"], color=f"C{k}", label=f"{lab} val")
        ax[1].plot(h["train_acc"], "--", color=f"C{k}")
        ax[1].plot(h["val_acc"], color=f"C{k}")
    ax[0].set_title("loss"); ax[1].set_title("accuracy"); ax[0].set_xlabel("epoch"); ax[1].set_xlabel("epoch")
    ax[0].legend()

In [ ]:
torch.manual_seed(0)
h_lin = fit(Linear(), train_loader, val_loader, epochs=4)
torch.manual_seed(0)
mlp = MLP()
h_mlp = fit(mlp, train_loader, val_loader, epochs=4)
plot_history(h_lin, h_mlp, labels=["linear", "MLP"])
assert h_mlp["val_acc"][-1] > h_lin["val_acc"][-1] + 0.01

**Questions.**
1. The MLP has ~300× more parameters than the linear model. Why does it generalise better and not just overfit?
2. Look at the predictions below: are the errors "reasonable"?

In [ ]:
x, y = next(iter(val_loader))
with torch.no_grad():
    pred = mlp.eval()(x.to(device)).argmax(1).cpu()
wrong = (pred != y).nonzero().ravel()[:8]
fig, ax = plt.subplots(1, 8, figsize=(16, 2.6))
for a_, k in zip(ax, wrong):
    a_.imshow(x[k, 0], cmap="gray"); a_.axis("off")
    a_.set_title(f"pred {classes[pred[k]]}\ntrue {classes[y[k]]}", fontsize=8)

## 6. Overfitting

Train the same MLP on only **1 000** images for 40 epochs and watch the curves.

In [ ]:
small = torch.utils.data.Subset(train_set, range(1000))
small_loader = torch.utils.data.DataLoader(small, batch_size=64, shuffle=True)
torch.manual_seed(0)
h_small = fit(MLP(), small_loader, val_loader, epochs=40)
plot_history(h_small, labels=["MLP, 1000 images"])

**Questions.**
1. At which epoch would you stop training? What is this technique called?
2. Name three other ways to reduce overfitting. (We will use them on CNNs this afternoon.)

## 7. Save and reload a model

Save the **`state_dict`** (the dictionary of weights), not the whole Python object. To reload, rebuild the same architecture and load the weights.

In [ ]:
torch.save(mlp.state_dict(), "mlp_fashion.pt")
reloaded = MLP()
reloaded.load_state_dict(torch.load("mlp_fashion.pt", map_location="cpu"))
loss_fn = nn.CrossEntropyLoss()
print("reloaded model, val accuracy:", evaluate(reloaded.to(device), val_loader, loss_fn)[1])